# Task 12: Knowledge Distillation of Dual-Encoder Vision-Language Models

## Objective

To compress a vision-language model into a lightweight student model using cosine similarity and KL-divergence knowledge distillation.

## Technologies / Tools Used

- Python 3.10+
- PyTorch
- Hugging Face Transformers
- TorchVision
- SciPy
- Google Colab

## Formula

### Cosine Loss

L_cos = 1 - cosine(Student, Teacher)

### KL-Divergence

L_KL = KL(Teacher || Student)

### Combined Loss

L = αL_cos + βL_KL

## Step 1: Load the CLIP Teacher Model

Load a pretrained CLIP model as the frozen teacher.

In [1]:
# Install required libraries

!pip -q install transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

# Load teacher

teacher = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False

print("CLIP teacher loaded.")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP teacher loaded.


## Step 2: Create a Lightweight Student Model

Create a small student network that learns to reproduce the teacher's image embeddings.

In [2]:
# Lightweight student model

class StudentModel(nn.Module):

    def __init__(self, input_dim=512, hidden_dim=128):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        return self.network(x)


student = StudentModel()

optimizer = torch.optim.Adam(
    student.parameters(),
    lr=0.001
)

print("Student model created.")

Student model created.


## Step 3: Knowledge Distillation

Use cosine similarity and KL-divergence to train the student using teacher representations.

In [3]:
# Example teacher representation

teacher_features = torch.randn(8, 512)

# Student prediction

student_features = student(teacher_features)

# Normalize features

teacher_norm = F.normalize(
    teacher_features,
    dim=-1
)

student_norm = F.normalize(
    student_features,
    dim=-1
)

# Cosine loss

cos_loss = (
    1 - F.cosine_similarity(
        student_norm,
        teacher_norm
    ).mean()
)

# KL divergence

teacher_prob = F.softmax(
    teacher_features,
    dim=-1
)

student_log_prob = F.log_softmax(
    student_features,
    dim=-1
)

kl_loss = F.kl_div(
    student_log_prob,
    teacher_prob,
    reduction="batchmean"
)

# Combined loss

loss = 0.7 * cos_loss + 0.3 * kl_loss

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Cosine Loss:", cos_loss.item())
print("KL Loss:", kl_loss.item())
print("Total Loss:", loss.item())

Cosine Loss: 0.9826171398162842
KL Loss: 0.5091388821601868
Total Loss: 0.8405736684799194


## Conclusion

A lightweight student model was trained using knowledge distillation from a frozen CLIP teacher. The combined loss used cosine similarity and KL-divergence to transfer representation knowledge to the student model.